In [1]:
from SmartApi import SmartConnect
from SmartApi.smartWebSocketV2 import SmartWebSocketV2
from strategy2 import BreakoutStrategy
import pyotp
import threading
import time
import datetime
import logging
import sys
api_key = '2a6JiSY1'
username = 'AAAB276031'
pwd = '2631'
smartApi = SmartConnect(api_key)
logging.basicConfig(filename='trade_logs.log', level=logging.INFO, format='%(asctime)s - %(message)s')
def login():
    global authToken , feedToken , refreshToken
    try:
        token = "XMWFDVJKMIWK2ZFAYC6SVPYN4Y"
        totp = pyotp.TOTP(token).now()
    except Exception as e:
        logging.error("Invalid Token: The provided token is not valid.")
        print(e)
        return False
    correlation_id = "abcde"
    data = smartApi.generateSession(username, pwd, totp)

    if data['status'] == False:
        print("Error in data: " , data)
        return False
    else:
        # login api call
        # logger.info(f"You Credentials: {data}")
        authToken = data['data']['jwtToken']
        refreshToken = data['data']['refreshToken']
        # fetch the feedtoken
        feedToken = smartApi.getfeedToken()
        # fetch User Profile
        res = smartApi.getProfile(refreshToken)
        smartApi.generateToken(refreshToken)
        res=res['data']['exchanges']
        print("Login successufu;")
    return True

[I 250701 09:55:20 smartConnect:121] in pool


In [2]:
def logout():
        try:
                logging.info("Initiating shutdown...")
                smartApi.terminateSession(username)
                logging.info("Cleanup complete. Exiting.")
                print("logout succeful")
        except Exception as e:
                logging.warning(f"Error during shutdown: {e}")

In [ ]:
status = "OK"
high = -1
stoploss = -1
sws = None
waiting_for_candle_update = -1
trail_anchor = -1
lock = threading.Lock()
def run_trading_loop():
    global high , stoploss , waiting_for_candle_update , trail_anchor
    global status
    global sws
    status = "OK"
    def shutdown(h , s , w , t):
        global status
        global high , stoploss , waiting_for_candle_update , trail_anchor
        try:
            logging.info("Initiating shutdown...")
            sws.close_connection()
            smartApi.terminateSession(username)
            logging.info("Cleanup complete. Exiting.")
        except Exception as e:
            logging.warning(f"Error during shutdown: {e}")
        high = h
        stoploss = s
        waiting_for_candle_update = w
        if(t==-1):
            trail_anchor = -1
        else:
            trail_anchor = datetime.datetime.strptime(t, "%Y-%m-%d %H:%M:%S")
        return
    def wait_until_946():
        global status
        now = datetime.datetime.now().time()
        print("Current time:", now)
        print("Waiting until 9:46 AM...")
        while True:
            now = datetime.datetime.now().time()
            if now >= datetime.time(15, 10):
                print("Too late to start. Exiting.")
                status = "OVER"
                logging.info("Too late to start. Exiting.")
                return
            if now < datetime.time(9, 46):
                time.sleep(30)
            else:
                print("Time crossed!")
                return

    waiting = wait_until_946()
    if status == "OVER":
        shutdown(high , stoploss , waiting_for_candle_update , trail_anchor)
        return 
    print("Entering market now...")
    # --- Market Check ---
    def is_market_open():
        try:
            ltp = smartApi.ltpData(exchange="NSE", tradingsymbol="NIFTYBEES-EQ", symboltoken="10576")
            return ltp.get("data") is not None
        except:
            return False

    if not is_market_open():
        print("Market seems closed. Exiting.")
        shutdown(high , stoploss , waiting_for_candle_update , trail_anchor)
        return 
    else:
        print("Market is opened. Proceeding...")
    EXCHANGE = "NSE"
    TRADINGSYMBOL = "NIFTYBEES-EQ"
    TOKEN = "10576"  # Int not str
    quantity = 50

    def safe_ltp():
        try:
            data = smartApi.ltpData(exchange=EXCHANGE, tradingsymbol=TRADINGSYMBOL, symboltoken=str(TOKEN))
            return float(data['data']['ltp']) if data and data.get('data') else None
        except Exception as e:
            print(f"LTP fetch failed: {e}")
            return None

    ltp = safe_ltp()
    if ltp is None:
        print("Unable to get ltp..")
        status = None
        shutdown(high , stoploss , waiting_for_candle_update , trail_anchor)
        return
    else:
        print("LTP fetched:", ltp)
    strategy = BreakoutStrategy(smartApi, TRADINGSYMBOL, EXCHANGE, str(TOKEN))
    print("Strategy Initialized and Intital position : " , strategy.position)
    # Restore state
    if high != -1 and stoploss != -1:
        strategy.high = high
        strategy.stoploss = stoploss
        strategy.waiting_for_candle_update = waiting_for_candle_update
        if trail_anchor!=-1:
            strategy.trail_anchor = trail_anchor
        else:
            now = datetime.datetime.now() - datetime.timedelta(minutes=1)
            aligned_minute = (now.minute // 30) * 30
            to_dt = now.replace(minute=aligned_minute, second=0, microsecond=0)
            from_dt = to_dt - datetime.timedelta(minutes=30)
            strategy.trail_anchor = from_dt
        print("Restored high and stoploss")
    else:
        print("Strategy initialized. Fetching first candle...")
        if not strategy.initialize_first_candle():
            print("First candle not ready. Waiting for 60 seconds...")
            time.sleep(60)
            if not strategy.initialize_first_candle():
               status = None
               shutdown(high , stoploss , waiting_for_candle_update , trail_anchor)
               return
            else:
                print("Candle fetched successfully after retry.")
        else:
            print("Initial candle fetched successfully.")
    print("Initializing WebSocket...")
    sws = SmartWebSocketV2(auth_token=authToken, api_key=api_key, client_code=username, feed_token=feedToken, max_retry_attempt=0)
    sws.auto_reconnect = False
    buy_order_id = None
    sell_order_id = None
    pending_buy = False
    pending_sell = False
    order_monitor_running = False
    global lock
    correlation_id = "abc123"
    mode = 1  # Full mode (tick data)
    exchangeType = 1  # NSE
    EXCHANGE = "NSE"
    TRADINGSYMBOL = "NIFTYBEES-EQ"
    NSE_TOKEN_NIFTYBEES = "10576"
    def place_order(order_type, price):
        nonlocal buy_order_id, sell_order_id, pending_buy, pending_sell
        try:
            order = smartApi.placeOrderFullResponse({
                "variety": "NORMAL",
                "tradingsymbol": TRADINGSYMBOL,
                "symboltoken": str(TOKEN),
                "transactiontype": order_type,
                "exchange": EXCHANGE,
                "ordertype": "LIMIT",
                "producttype": "INTRADAY",
                "duration": "DAY",
                "price": price,
                "quantity": quantity
            })
            oid = order['data']['orderid']
            if order_type == "BUY":
                buy_order_id = oid
                pending_buy = True
            else:
                sell_order_id = oid
                pending_sell = True
            print(f"{order_type} order placed at limit {price}. ID: {oid}")
            return
        except Exception as e:
            print(f"{order_type} order failed: {e}")
            return
    def monitor_order(order_id, is_buy):
        nonlocal strategy
        nonlocal pending_buy, pending_sell, order_monitor_running
        if order_monitor_running:
            print("order running already")
            shutdown(strategy.high , strategy.stoploss , strategy.waiting_for_candle_update , strategy.trail_anchor)
            return
        order_monitor_running = True
        start = time.time()
        while time.time() - start < 300:
            try:
                orders = smartApi.orderBook()
                for order in orders.get('data', []):
                    if order['orderid'] == order_id and order['status'] == 'complete':
                        if is_buy:
                            strategy.position = True
                            pending_buy = False
                            strategy.waiting_for_candle_update = False
                        else:
                            strategy.position = False
                            pending_sell = False
                            strategy.waiting_for_candle_update = True
                        print(f"{'Buy' if is_buy else 'Sell'} order executed , position: {strategy.position}")
                        order_monitor_running = False
                        return
            except Exception as e:
                print(f"Order monitoring failed: {e}")
                return shutdown(strategy.high , strategy.stoploss , strategy.waiting_for_candle_update , strategy.trail_anchor)
            time.sleep(10)
        smartApi.cancelOrder(orderid=order_id, variety="NORMAL")
        pending_buy = pending_sell = False
        order_monitor_running = False
        print("Order not filled in 5 minutes. Cancelled.")
        return
    def on_data(wsapp, message):
        global status
        nonlocal pending_buy, pending_sell
        nonlocal strategy
        try:
            ltp = message.get('last_traded_price')/100
            now = datetime.datetime.now().time()
            if strategy.force_exit_required():
                if strategy.position:
                    place_order("SELL", ltp)
                    threading.Thread(target=monitor_order, args=(sell_order_id, False)).start()
                print("EOD reached. Shutting down.")
                return shutdown(strategy.high , strategy.stoploss , strategy.waiting_for_candle_update , strategy.trail_anchor)
            with lock:
                k = strategy.is_high_breached(ltp)
                if strategy.should_trail_stoploss():
                    print("Should Trail Stoploss.")
                    if strategy.update_trailing_stoploss(ltp) is None:
                        status = None
                        shutdown(strategy.high , strategy.stoploss , strategy.waiting_for_candle_update , strategy.trail_anchor)
                    return
                if (k==True) and (not pending_buy) and (not strategy.waiting_for_candle_update):
                    confirm_ltp = safe_ltp()
                    breakout_confirm = strategy.confirm_breakout(confirm_ltp)
                    if(breakout_confirm) is None:
                        status = None
                        return shutdown(strategy.high , strategy.stoploss , strategy.waiting_for_candle_update , strategy.trail_anchor)
                    if confirm_ltp and breakout_confirm:
                        print("confirm_ltp : " , confirm_ltp)
                        print("placing buy order now..")
                        place_order("BUY", confirm_ltp*(1.01))
                        threading.Thread(target=monitor_order, args=(buy_order_id, True)).start()
                        return
                    return
                elif strategy.position and strategy.stoploss_hit() and not pending_sell:
                    print("Stop loss is hit. Placing Sell order..")
                    place_order("SELL", ltp*(0.99))
                    threading.Thread(target=monitor_order, args=(sell_order_id, False)).start()
                    strategy.waiting_for_candle_update = True
                    return
                elif strategy.waiting_for_candle_update:
                    now = datetime.datetime.now()
                    if now.minute %30 == 15 :
                        if strategy.update_high_stoploss_after_sell() is None:
                            return shutdown(strategy.high , strategy.stoploss , strategy.waiting_for_candle_update , strategy.trail_anchor)
                    return
                return
        except Exception as e:
            logging.error(f"on_data error: {e}")
            return
    def on_open(wsapp):
        logging.info("WebSocket connected.")
        try:
            print("WebSocket connection opened.")
            res = sws.subscribe(correlation_id, mode, [{
                "exchangeType": exchangeType,
                "tokens": [NSE_TOKEN_NIFTYBEES]
            }])
            print("Successfully subscribed:", res)
            print("High: " , strategy.high)
            print("StopLoss: " , strategy.stoploss)
            print("ltp: " , ltp)
            print("achnor time: "  , strategy.trail_anchor)
            return
        except Exception as e:
            logging.error(f"Subscribe failed: {e}")
            return

    def on_error(wsapp, error):
        logging.error(f"WebSocket error: {error}")
        return

    def on_close(wsapp):
        logging.info("WebSocket closed.")
        return

    # Assign WebSocket handlers
    sws.on_open = on_open
    sws.on_data = on_data
    sws.on_error = on_error
    sws.on_close = on_close


    # --- Start WebSocket in Main Thread ---
    print("Trying to connect WebSocket...")
    sws.connect()
def run():
    global status
    global high , stoploss , waiting_for_candle_update
    high = -1
    stoploss = -1
    waiting_for_candle_update = False
    while True:
        print("Running the function now..")
        now = datetime.datetime.now().time()
        if(now >=datetime.time(15, 10)):
            print("Trading session over..")
            return
        result = login()
        if(not result):
            print("Trying to Login in again in 2min..")
            time.sleep(120)
            continue
        run_trading_loop()
        print("Loop over.")
        logout()
        if status is None:
            print("Restarting in 2 min...")
            time.sleep(120)
            continue
        else:
            print("Trading session over..")
            return
run()

Running the function now..


Login successufu;
Current time: 09:55:35.554822
Waiting until 9:46 AM...
Time crossed!
Entering market now...
Market is opened. Proceeding...
LTP fetched: 287.1
Strategy Initialized and Intital position :  False
Strategy initialized. Fetching first candle...
Initial high: 287.89, stoploss: 286.31
Initial candle fetched successfully.
Initializing WebSocket...
Trying to connect WebSocket...
WebSocket connection opened.
Successfully subscribed: None
High:  287.89
StopLoss:  286.31
ltp:  287.1
achnor time:  2025-07-01 09:00:00
EOD reached. Shutting down.


[W 250701 15:10:03 smartWebSocketV2:343] Connection closed due to max retry attempts reached.


Loop over.
logout succeful
Trading session over..


In [4]:
logout()

logout succeful
